# Model-stage preregistration

Ten notebook prerejestruje etap modelowy **przed jakimkolwiek treningiem**. Jego wejściami są wyłącznie zamrożone specyfikacje i manifesty. Nie wczytuje model matrix, target values ani danych analitycznych z external validation 2021–2022 lub test 2023–2024.

Zakres obejmuje registry modeli, search spaces, budżety, seedy, QNN architecture search, regułę selekcji, kalibrację, threshold, interpretability oraz PIT-safe validation/test policy. Supervised ML pipeline v1.0.0 i wszystkie upstream frozen artifacts pozostają immutable.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
import yaml

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 240)

def project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs/supervised_ml_pipeline_v1_freeze_manifest.yaml").is_file():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu głównego projektu.")

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

ROOT = project_root()
EXPECTED_HASHES = {
    "configs/target_candidate_v2_pit_b_freeze_manifest.yaml": "52fd67d360e486e45615330a869f8b7d5810eb08d957432b4c2da7cc146b66bb",
    "configs/research_universe_pit_freeze_manifest.yaml": "60310dbc9379371c05316b28de273832d0eaf02f20fc1ee7bb28697a26fb71b7",
    "configs/x_t_pit_v1_freeze_manifest.yaml": "9b59e812bfb1b34a2f72c78ce4fc0ba484249d0a1d48cdea8f94506a403a9023",
    "configs/supervised_ml_pipeline_v1.yaml": "0e817dac719d1651ec7518141e71c627dc208f4bed5ccfebed6c5b9d88652765",
    "configs/supervised_ml_pipeline_v1_freeze_manifest.yaml": "f1000d9e66a83160ff4ae0c5759c09c96491e18c4b579f6a397e0e98afc6eef1",
    "docs/04_9_target_candidate_v2_pit_b_frozen_specification.md": "029d4ab5040d96fd3856c5d3ef27b0deb96d34f0ebdde247e2e8afd8e605a002",
    "docs/07_1_supervised_ml_pipeline_v1_frozen_specification.md": "5dcdeeb037de81f92ab1fae393005c487c8dd3a0edfe80997c49ded2d3eaa7dc",
    "configs/model_stage_candidates_v1.json": "857ed6361a55c4ff1183a56614a4058e132db7a8b49bb5742b187900cd9d7f58",
}
actual_hashes = {relative: sha256(ROOT / relative) for relative in EXPECTED_HASHES}
assert actual_hashes == EXPECTED_HASHES, "Frozen input hash mismatch."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.modeling.model_stage_preregistration import canonical_sha256, materialized_registry
candidate_registry = json.loads((ROOT / "configs/model_stage_candidates_v1.json").read_text())
assert candidate_registry == materialized_registry(), "Materialized candidate lists differ from deterministic generator."

target_manifest = yaml.safe_load((ROOT / "configs/target_candidate_v2_pit_b_freeze_manifest.yaml").read_text())
universe_manifest = yaml.safe_load((ROOT / "configs/research_universe_pit_freeze_manifest.yaml").read_text())
x_t_manifest = yaml.safe_load((ROOT / "configs/x_t_pit_v1_freeze_manifest.yaml").read_text())
pipeline = yaml.safe_load((ROOT / "configs/supervised_ml_pipeline_v1.yaml").read_text())
pipeline_manifest = yaml.safe_load((ROOT / "configs/supervised_ml_pipeline_v1_freeze_manifest.yaml").read_text())

frozen_identities = [
    (target_manifest["target"], "target_candidate_v2_pit_b", "1.0.0"),
    (universe_manifest["historical_research_universe"], "research_universe_pit", "1.1.0"),
    (x_t_manifest["raw_point_in_time_x_t"], "x_t_pit", "1.0.0"),
    (pipeline_manifest["supervised_ml_pipeline"], "supervised_ml_pipeline", "1.0.0"),
]
for identity, expected_id, expected_version in frozen_identities:
    assert identity["id"] == expected_id
    assert identity["version"] == expected_version
    assert identity["status"] == "frozen"
assert pipeline_manifest["supervised_ml_pipeline"]["predictive_models_trained"] is False

pd.DataFrame(
    [{"artifact": identity["id"], "version": identity["version"], "status": identity["status"]}
     for identity, _, _ in frozen_identities]
)

## 1. Immutable experiment contract

Main sample, preprocessing C, feature blocks, temporal CV, primary metric, clustered inference i robustness checks pochodzą bez zmian z pipeline v1.0.0. Search korzysta wyłącznie z OOF 2015–2020. External validation i test nie mogą aktywować nowych prób ani refinement.

In [ ]:
EXPERIMENT_CONTRACT = {
    "sample": {
        "target_status": "available",
        "x_t_status": ["available_core", "partially_available"],
        "development_n": 23218,
        "train_n": 19671,
        "external_validation_n": 3547,
    },
    "feature_blocks": ["L", "L+D", "L+D+R"],
    "preprocessing": [
        "fold_train_p1_p99_winsorization",
        "fold_train_median_imputation",
        "per_feature_missing_indicators",
        "financial_feature_standard_scaling",
        "binary_indicators_unscaled",
    ],
    "primary_metric": "pooled_oof_pr_auc_2015_2020",
    "fold_reporting": ["pr_auc_each_fold", "mean", "sample_sd"],
    "bootstrap": {"cluster": "economic_group_id", "replicates": 2000, "ci": .95, "seed": 20260818},
    "model_training_performed": False,
}
FOLDS = pipeline["temporal_cv"]["folds"]
assert len(FOLDS) == 6
assert [fold["validation_feature_years"][0] for fold in FOLDS] == list(range(2015, 2021))
assert pipeline["metric_aggregation"]["primary_ranking"]["name"] == "pooled_oof_pr_auc"
assert pipeline["temporal_cv"]["target_availability_rule"] == "target_available_at <= min(prediction_timestamp_in_validation_fold)"
pd.DataFrame(FOLDS)[["id", "train_feature_years", "embargo_feature_years", "validation_feature_years", "pit_safe_train_n", "validation_n"]]

## 2. Model roster i spójność z dokumentacją

Wcześniejszy dokument legacy deklarował `XGBoost/LightGBM`; dlatego XGBoost wchodzi jawnie do benchmarku, a LightGBM jest pominięty jako alternatywna implementacja tego samego punktu. Repozytorium nie zawiera wcześniejszej deklaracji LDA ani pojedynczego Decision Tree. LDA jest pominięte jako redundantny linear baseline o problematycznym założeniu wspólnej macierzy kowariancji dla macierzy z indicators. Pojedyncze drzewo jest pominięte z powodu wysokiej wariancji w temporal folds; tree family reprezentują RF, HistGB i XGBoost.

Żaden model nie może omijać frozen preprocessing C.

In [ ]:
MODEL_REGISTRY = [
    {"id": "dummy_prior", "role": "prevalence floor", "blocks": "block-agnostic",
     "space": {"strategy": ["prior"]}, "search": "grid", "main_trials_per_block": 1, "confirmation": "none"},
    {"id": "fixed_l2_logistic", "role": "simple statistical baseline", "blocks": "all",
     "space": {"C": [1.0], "solver": ["lbfgs"], "max_iter": [2000], "tol": [1e-6], "imbalance": ["none", "sqrt"]},
     "search": "exhaustive", "main_trials_per_block": 2, "confirmation": "none"},
    {"id": "elastic_net_logistic", "role": "regularized linear model", "blocks": "all",
     "space": {"penalty": ["elasticnet"], "C": [1e-3, 1e-2, 1e-1, 1, 10, 100], "l1_ratio": [0, .25, .5, .75, 1],
               "solver": ["saga"], "max_iter": [5000], "tol": [1e-4], "imbalance": ["none", "sqrt"]},
     "search": "random_without_replacement", "main_trials_per_block": 12, "confirmation": "top_2_two_extra_seeds"},
    {"id": "rbf_svm", "role": "classical nonlinear ML", "blocks": "all",
     "space": {"kernel": ["rbf"], "C": [.1, 1, 10, 100], "gamma": ["scale", .01, .1],
               "probability": [False], "max_iter": [100000], "imbalance": ["none", "sqrt"]},
     "search": "random_without_replacement", "main_trials_per_block": 12, "confirmation": "deterministic"},
    {"id": "random_forest", "role": "bagging tree ensemble", "blocks": "all",
     "space": {"n_estimators": [600], "criterion": ["gini", "log_loss"], "max_depth": [None, 6, 12, 24],
               "min_samples_leaf": [1, 5, 20], "max_features": ["sqrt", .5, 1.0],
               "max_samples": [.7, 1.0], "bootstrap": [True], "imbalance": ["none", "sqrt"]},
     "search": "random_without_replacement", "main_trials_per_block": 12, "confirmation": "top_2_two_extra_seeds"},
    {"id": "hist_gradient_boosting", "role": "sklearn boosting", "blocks": "all",
     "space": {"learning_rate": [.03, .06, .1], "max_iter": [150, 300], "max_leaf_nodes": [7, 15, 31],
               "min_samples_leaf": [20, 50, 100], "l2_regularization": [0, .1, 1, 10],
               "max_features": [.7, 1.0], "early_stopping": [False], "imbalance": ["none", "sqrt"]},
     "search": "random_without_replacement", "main_trials_per_block": 16, "confirmation": "top_2_two_extra_seeds"},
    {"id": "xgboost", "role": "pre-registered external boosting", "blocks": "all",
     "space": {"tree_method": ["hist"], "objective": ["binary:logistic"], "n_estimators": [200, 500, 800],
               "max_depth": [2, 3, 5, 8], "learning_rate": [.02, .05, .1], "subsample": [.7, 1.0],
               "colsample_bytree": [.7, 1.0], "min_child_weight": [1, 5, 20], "reg_alpha": [0, .01, .1],
               "reg_lambda": [1, 5, 20], "gamma": [0, .1], "early_stopping": [False],
               "imbalance": ["none", "sqrt"]},
     "search": "random_without_replacement", "main_trials_per_block": 16, "confirmation": "top_2_two_extra_seeds"},
    {"id": "pytorch_mlp", "role": "classical neural benchmark", "blocks": "all",
     "space": {"hidden_layer_sizes": [(16,), (32,), (64,), (32, 16), (64, 32)], "activation": ["relu", "tanh"],
               "weight_decay": [1e-5, 1e-4, 1e-3], "learning_rate": [3e-4, 1e-3, 3e-3],
               "batch_size": [64, 256], "optimizer": ["Adam"], "epochs": [200],
               "early_stopping": [False], "imbalance": ["none", "sqrt"]},
     "search": "random_without_replacement", "main_trials_per_block": 12, "confirmation": "top_2_two_extra_seeds"},
]
MODEL_SPECIFIC_PREPROCESSING = {
    "dummy_prior": "none; feature matrix ignored",
    "fixed_l2_logistic": "none beyond frozen C",
    "elastic_net_logistic": "none beyond frozen C",
    "rbf_svm": "none beyond frozen C",
    "random_forest": "none beyond frozen C; frozen scaling is retained",
    "hist_gradient_boosting": "none beyond frozen C; frozen scaling is retained",
    "xgboost": "none beyond frozen C; frozen scaling is retained",
    "pytorch_mlp": "none beyond frozen C",
    "qnn": "fold-train-only PCA, PCA-component scaling, clipping to [-3,3], angle mapping to [-pi,pi] after frozen C",
}
DOCUMENTATION_DECISIONS = [
    {"model": "XGBoost", "prior_declaration": True, "decision": "included", "reason": "Declared as XGBoost/LightGBM in legacy methodology."},
    {"model": "LightGBM", "prior_declaration": "alternative", "decision": "omitted", "reason": "XGBoost instantiates the earlier alternative; avoids a third boosting implementation."},
    {"model": "LDA", "prior_declaration": False, "decision": "omitted", "reason": "Linear role covered by logistic models; covariance assumptions are unattractive with indicators."},
    {"model": "Decision Tree", "prior_declaration": False, "decision": "omitted", "reason": "High temporal-fold variance; tree family covered by RF, HistGB and XGBoost."},
]
ELASTIC_NET_CONSTRUCTOR = {"penalty": "elasticnet", "dual": False, "tol": 1e-4, "C": "candidate_value", "fit_intercept": True, "intercept_scaling": 1, "class_weight": None, "random_state": "training_seed", "solver": "saga", "max_iter": 5000, "verbose": 0, "warm_start": False, "n_jobs": 1, "l1_ratio": "candidate_value"}
benchmark_table = pd.DataFrame([{**{k: v for k, v in model.items() if k != "space"}, "space": json.dumps(model["space"], default=str), "model_specific_preprocessing": MODEL_SPECIFIC_PREPROCESSING[model["id"]]} for model in MODEL_REGISTRY])
qnn_registry_row = pd.DataFrame([{"id": "qnn", "role": "experimental challenger", "blocks": "all", "space": "controlled Stage Q1 plus full-configuration Stage Q2", "search": "two_stage_frozen_temporal_cv", "main_trials_per_block": 6, "confirmation": "top_1_two_extra_seeds", "model_specific_preprocessing": MODEL_SPECIFIC_PREPROCESSING["qnn"]}])
pd.concat([benchmark_table, qnn_registry_row], ignore_index=True)

### PyTorch MLP — frozen implementation contract

MLP nie używa `sklearn.neural_network.MLPClassifier`. Jest implementowany bezpośrednio w PyTorch, dzięki czemu loss, sqrt weighting, raw logits i Integrated Gradients mają jeden audytowalny interfejs. `alpha` zostało przełożone na Adam `weight_decay`, a `max_iter` na pełne `epochs`.

In [ ]:
PYTORCH_MLP_CONTRACT = {
    "module": "torch.nn.Sequential of Linear+activation hidden layers and one scalar Linear output",
    "dtype": "torch.float64",
    "dropout": False,
    "batch_normalization": False,
    "raw_output": "one_logit_per_observation",
    "weight_initialization": "Xavier uniform; hidden gain=calculate_gain(activation), output gain=1",
    "bias_initialization": 0.0,
    "initialization_seed": "training_seed_set_before_module_construction",
    "optimizer": {"class": "torch.optim.Adam", "betas": [0.9, 0.999], "eps": 1e-8, "amsgrad": False, "maximize": False, "foreach": False, "fused": False, "weight_decay": "candidate_value", "learning_rate": "candidate_value"},
    "loss": {"class": "torch.nn.BCEWithLogitsLoss", "pos_weight": "fold_train_sqrt_N_negative_over_N_positive_or_1", "reduction": "mean"},
    "epochs": {"coarse": 200, "refinement": 300},
    "early_stopping": False,
    "batch_loader": {"shuffle": True, "generator_seed": "training_seed*10000+validation_year", "drop_last": False, "num_workers": 0, "persistent_workers": False},
    "determinism": {"torch_use_deterministic_algorithms": True, "torch_num_threads": 1, "torch_num_interop_threads": 1, "cudnn_benchmark": False, "cudnn_deterministic": True},
    "checkpoint": {"atomic_write": True, "every_epochs": 5, "resume_limit": 1, "contents": ["model_state", "optimizer_state", "epoch", "rng_states", "configuration_id", "fold_id", "preprocessing_hash", "software_versions"]},
}
pd.Series(PYTORCH_MLP_CONTRACT, name="frozen_contract")

## 3. Conditional classical refinement

Po coarse search refinement kwalifikuje najwyżej trzy rodziny. Rodzina musi być nie dalej niż 0.010 pooled OOF PR-AUC od globalnego coarse classical/MLP leadera oraz spełnić co najmniej jeden warunek: najlepszy punkt leży na granicy uporządkowanego parametru albo odstęp do runner-up jest nie większy niż 0.003. Refinement dotyczy tylko najlepszego bloku rodziny. Reguła może używać wyłącznie CV 2015–2020.

In [ ]:
REFINEMENT_POLICY = {
    "maximum_families": 3,
    "coarse_leader_pr_auc_distance_max": .010,
    "runner_up_gap_max": .003,
    "boundary_or_close_runner_up_required": True,
    "eligible_data": "OOF_2015_2020_only",
    "best_feature_block_only": True,
    "validation_or_test_may_activate": False,
}
REFINEMENT_REGISTRY = [
    {"id": "elastic_net_logistic", "limit": 8, "space": {"C": [.003, .03, .3, 3, 30, 300], "l1_ratio": [.125, .375, .625, .875], "imbalance": ["none", "sqrt"]}},
    {"id": "rbf_svm", "limit": 8, "space": {"C": [.03, .3, 3, 30, 300], "gamma": [.003, .03, .3], "imbalance": ["none", "sqrt"]}},
    {"id": "random_forest", "limit": 8, "space": {"n_estimators": [800], "max_depth": [4, 9, 18, 32], "min_samples_leaf": [2, 10, 35], "max_features": [.35, .7], "max_samples": [.85], "criterion": ["gini", "log_loss"], "imbalance": ["none", "sqrt"]}},
    {"id": "hist_gradient_boosting", "limit": 10, "space": {"learning_rate": [.02, .045, .08], "max_iter": [225, 450], "max_leaf_nodes": [11, 23, 47], "min_samples_leaf": [10, 35, 75], "l2_regularization": [.03, .3, 3], "max_features": [.85], "imbalance": ["none", "sqrt"]}},
    {"id": "xgboost", "limit": 10, "space": {"n_estimators": [350, 650, 1000], "max_depth": [2, 4, 6], "learning_rate": [.015, .035, .075], "min_child_weight": [2, 10], "subsample": [.85], "colsample_bytree": [.85], "reg_alpha": [.03, .3], "reg_lambda": [2, 10], "gamma": [.03, .3], "imbalance": ["none", "sqrt"]}},
    {"id": "pytorch_mlp", "limit": 8, "space": {"hidden_layer_sizes": [(24,), (48,), (96,), (48, 24), (96, 48)], "activation": ["relu", "tanh"], "weight_decay": [3e-5, 3e-4, 3e-3], "learning_rate": [1e-4, 6e-4, 2e-3], "batch_size": [128], "epochs": [300], "imbalance": ["none", "sqrt"]}},
]
refinement_table = pd.DataFrame([{"model": row["id"], "limit": row["limit"], "space": json.dumps(row["space"], default=str)} for row in REFINEMENT_REGISTRY])
materialized_lists = {**{f"coarse.{key}": value for key, value in candidate_registry["coarse"].items()}, **{f"refinement.{key}": value for key, value in candidate_registry["refinement"].items()}, "qnn.stage_q1": candidate_registry["qnn"]["stage_q1"], "qnn.stage_q2": candidate_registry["qnn"]["stage_q2"]}
assert set(materialized_lists) == set(candidate_registry["list_hashes"])
assert all(canonical_sha256(materialized_lists[name]) == expected for name, expected in candidate_registry["list_hashes"].items())
configuration_ids = [candidate["configuration_id"] for candidates in materialized_lists.values() for candidate in candidates]
assert len(configuration_ids) == len(set(configuration_ids))
display_hashes = pd.DataFrame([{"candidate_list": name, "n": len(materialized_lists[name]), "sha256": digest} for name, digest in candidate_registry["list_hashes"].items()])
display_hashes

## 4. Dwuetapowy QNN architecture search

QNN działa w osobnym Pythonie 3.12 z dokładnym dependency lockiem. Primary device to PennyLane `default.qubit`, `shots=None`, float64, local noiseless simulation i adjoint differentiation. Reprezentacja to frozen preprocessing C, train-fold-only PCA, train-fold-only scaling komponentów, clipping do [-3,3] i mapowanie do [-π,π].

### Stage Q1: kontrolowany wybór ansatzu

Trzy ansatze mają identyczne pozostałe ustawienia: 4 qubity/PCA, 2 warstwy, 45 epok, learning rate .01, batch 128, weight decay 1e-4 i imbalance sqrt. Stage Q1 jest **primary wewnątrz QNN**: osobno na każdym bloku wybiera ansatz według pooled OOF PR-AUC. Stage Q2 nie może go później zastąpić. Możliwe interactions ansatz–hyperparameters są raportowanym ograniczeniem.

In [ ]:
QNN_ANSATZES = {
    "ROT_CNOT_RING": "Rot on every wire followed by a directed CNOT ring",
    "RY_RZ_CZ_BRICKWORK": "RY and RZ on every wire followed by alternating even-odd/odd-even CZ brickwork",
    "RY_CRX_RING": "RY on every wire followed by trainable CRX gates on ring edges",
}
QNN_COMMON_MODEL = {
    "encoding": "ascending-wire RY angle encoding once before variational layers",
    "measurements": "PauliZ_expectation_on_every_qubit",
    "head": "single_trainable_linear_layer_from_all_expectations_to_one_logit",
    "optimizer": "Adam",
    "loss": "torch.nn.BCEWithLogitsLoss(reduction='mean')",
    "gradient_clip_norm": 5,
    "early_stopping": False,
    "circuit_parameter_initialization": "iid Uniform[-0.1,0.1] from training_seed",
    "linear_head_initialization": "Xavier uniform gain=1 from same seeded generator; bias=0",
    "parameter_order": "layer-major, then ascending wire, then gate parameter; head weights wire 0..q-1, then bias",
}
PCA_QNN_CONTRACT = {
    "input_source": "FinancialPreprocessor transformed output",
    "input_order": "all financial features in frozen block order, followed by one __missing indicator per feature in the same order",
    "includes_missing_indicators": True,
    "feature_orders": candidate_registry["pca_feature_order"],
    "pca": {"class": "sklearn.decomposition.PCA", "n_components": "candidate qubits 4 or 6", "svd_solver": "full", "whiten": False, "copy": True, "random_state": None},
    "component_scaling": {"class": "sklearn.preprocessing.StandardScaler", "with_mean": True, "with_std": True, "copy": True, "fit_scope": "fold_train_only_after_PCA"},
    "clipping": [-3.0, 3.0],
    "angle_mapping": "angle=(pi/3)*clip(scaled_component,-3,3), giving [-pi,pi]",
}
QNN_ARCHITECTURE_PACKAGES = {
    "ROT_CNOT_RING": {"trainable_parameters_q4_depth2_including_head": 29, "layer_gate_order": "Rot(phi,theta,omega) on wires 0..q-1; then CNOT control=i,target=(i+1)%q for i=0..q-1"},
    "RY_RZ_CZ_BRICKWORK": {"trainable_parameters_q4_depth2_including_head": 21, "layer_gate_order": "RY then RZ on wires 0..q-1; even layer CZ(0,1),(2,3),...; odd layer CZ(1,2),(3,4),...,(q-1,0)"},
    "RY_CRX_RING": {"trainable_parameters_q4_depth2_including_head": 21, "layer_gate_order": "RY on wires 0..q-1; then CRX control=i,target=(i+1)%q for i=0..q-1"},
}
QNN_STAGE_Q1 = {
    "ansatzes": list(QNN_ANSATZES),
    "qubits_pca_components": 4,
    "layers": 2,
    "epochs": 45,
    "learning_rate": .01,
    "batch_size": 128,
    "weight_decay": 1e-4,
    "imbalance": "sqrt",
    "optimizer": QNN_COMMON_MODEL["optimizer"],
    "loss": QNN_COMMON_MODEL["loss"],
    "gradient_clip_norm": QNN_COMMON_MODEL["gradient_clip_norm"],
    "seed": 20260818,
    "shared_minibatch_order": "generator_seed=training_seed*10000+validation_year; identical row order and batches for all three packages",
    "selection_scope": "separate_winner_per_feature_block",
    "may_select_final_ansatz": True,
    "stage_q2_may_change_ansatz": False,
    "tie_break": ["pooled_OOF_PR_AUC_rounded_to_6_decimals", "fewer_trainable_parameters", "fixed_before_trainable_entanglers", "lexicographic_ansatz_id"],
    "interpretation": "comparison_of_architecture_packages_not_a_causal_effect_of_the_entangling_gate",
}
QNN_STAGE_Q2 = [
    {"id": "T0", "qubits_pca": 4, "layers": 2, "epochs": 45, "learning_rate": .01, "batch_size": 128, "weight_decay": 1e-4, "imbalance": "sqrt", "reuse_q1_winner": True},
    {"id": "T1", "qubits_pca": 4, "layers": 1, "epochs": 30, "learning_rate": .03, "batch_size": 256, "weight_decay": 0, "imbalance": "none", "reuse_q1_winner": False},
    {"id": "T2", "qubits_pca": 6, "layers": 2, "epochs": 45, "learning_rate": .01, "batch_size": 128, "weight_decay": 1e-4, "imbalance": "sqrt", "reuse_q1_winner": False},
    {"id": "T3", "qubits_pca": 6, "layers": 3, "epochs": 60, "learning_rate": .003, "batch_size": 256, "weight_decay": 1e-3, "imbalance": "none", "reuse_q1_winner": False},
]
QNN_SENSITIVITY = [
    "replace_entangling_gates_with_identity",
    "replace_with_first_nonselected_ansatz_at_fixed_final_settings",
    "replace_with_second_nonselected_ansatz_at_fixed_final_settings",
    "swap_4_and_6_qubit_pca_at_fixed_other_settings",
]
PCA_MATCHED_CONTROLS = {
    "scope": "global QNN representative only, same fold-fitted PCA/scaler/clipping/angles and same rows",
    "fixed_logistic": {"C": 1.0, "penalty": "l2", "solver": "lbfgs", "imbalance": "same_as_selected_QNN", "seed": None},
    "pytorch_mlp": {"hidden_layer_sizes": "(2*q,)", "activation": "relu", "epochs": "same_as_selected_QNN", "learning_rate": 1e-3, "batch_size": "same_as_selected_QNN", "weight_decay": 1e-4, "imbalance": "same_as_selected_QNN", "seed": 20260818},
    "role": "diagnostic_only_no_primary_ranking_no_QNN_selection",
    "budget_fold_fits": {"fixed_logistic": 6, "pytorch_mlp": 6, "total": 12},
}
q1_table = pd.DataFrame([{"ansatz": key, "definition": value, **{k: v for k, v in QNN_STAGE_Q1.items() if k not in {"ansatzes", "selection_scope"}}} for key, value in QNN_ANSATZES.items()])
q2_table = pd.DataFrame(QNN_STAGE_Q2)
pd.concat({"controlled_ansatz_stage": q1_table, "full_configuration_stage": q2_table}, names=["qnn_stage", "row"])

## 5. Budżety, seedy i QNN resource policy

Niewykorzystany budżet nie może zostać przeniesiony na nowe konfiguracje. Przekroczenie limitu nie może prowadzić do uproszczenia ansatzu, liczby qubitów, depth lub liczby epok.

In [ ]:
SEEDS = {
    "candidate_sampling": 20260818,
    "refinement_sampling": 20260821,
    "training_and_confirmation": [20260818, 20260819, 20260820],
    "interpretability": 20260818,
    "clustered_bootstrap": 20260818,
}
MODEL_FOLD_FIT_BUDGETS = {
    "dummy_prior": 6,
    "fixed_l2_logistic": 36,
    "elastic_net_logistic": 288,
    "rbf_svm": 216,
    "random_forest": 288,
    "hist_gradient_boosting": 360,
    "xgboost": 360,
    "pytorch_mlp": 288,
}
QNN_BUDGET = {
    "stage_q1": 54,
    "stage_q2_new_configurations": 54,
    "top_1_per_block_two_seed_confirmation": 36,
    "interpretability_and_sensitivity": 24,
    "frozen_pipeline_robustness_if_global_winner": 30,
    "frozen_label_robustness_if_global_winner": 18,
    "retry_reserve": 24,
}
DIAGNOSTIC_BUDGET = {"pca_matched_fixed_logistic_fold_fits": 6, "pca_matched_pytorch_mlp_fold_fits": 6, "total": 12, "included_in_primary_ranking_budget": False, "included_in_QNN_240_cap": False}
SEARCH_BUDGET = {
    "classical_mlp_main": sum(MODEL_FOLD_FIT_BUDGETS.values()),
    "qnn_architecture_selection": QNN_BUDGET["stage_q1"] + QNN_BUDGET["stage_q2_new_configurations"] + QNN_BUDGET["top_1_per_block_two_seed_confirmation"],
    "main_model_selection": sum(MODEL_FOLD_FIT_BUDGETS.values()) + 144,
    "conditional_classical_refinement": 180,
    "ranking_search_cap": sum(MODEL_FOLD_FIT_BUDGETS.values()) + 144 + 180,
}
assert SEARCH_BUDGET == {"classical_mlp_main": 1842, "qnn_architecture_selection": 144, "main_model_selection": 1986, "conditional_classical_refinement": 180, "ranking_search_cap": 2166}
assert sum(QNN_BUDGET.values()) == 240
model_budget_table = pd.DataFrame.from_dict(MODEL_FOLD_FIT_BUDGETS, orient="index", columns=["maximum_fold_fits"])
qnn_budget_table = pd.DataFrame.from_dict(QNN_BUDGET, orient="index", columns=["maximum_fit_attempts"])
pd.concat({"model_search": model_budget_table, "qnn_total": qnn_budget_table}, names=["budget_group", "item"])

In [ ]:
ENVIRONMENT_CONTRACT = {
    "classical": {"python": "3.13.13", "numpy": "2.4.4", "scipy": "1.17.1", "pandas": "3.0.3", "scikit-learn": "1.8.0", "xgboost": "3.4.1", "shap": "0.52.0", "joblib": "1.5.3", "threadpoolctl": "3.6.0"},
    "qnn_mlp": {"python": "3.12.2", "numpy": "2.4.4", "scipy": "1.17.1", "pandas": "3.0.3", "scikit-learn": "1.8.0", "torch": "2.13.0", "PennyLane": "0.45.1", "pennylane-lightning": "0.45.0", "captum": "0.9.0"},
    "threads": {"OMP_NUM_THREADS": 1, "MKL_NUM_THREADS": 1, "OPENBLAS_NUM_THREADS": 1, "VECLIB_MAXIMUM_THREADS": 1, "NUMEXPR_NUM_THREADS": 1, "sklearn_n_jobs": 1, "xgboost_n_jobs": 1, "torch_num_threads": 1, "torch_num_interop_threads": 1},
}
CLASS_WEIGHTING_CONTRACT = {
    "formula": {"w_negative": 1.0, "w_positive": "sqrt(N_negative/N_positive)", "count_scope": "current train fold only", "validation_weighted": False},
    "dummy_prior": {"api": "none", "imbalance_modes": ["none"]},
    "fixed_l2_logistic": {"api": "fit(sample_weight=vector)", "class_weight": None},
    "elastic_net_logistic": {"api": "fit(sample_weight=vector)", "class_weight": None},
    "rbf_svm": {"api": "fit(sample_weight=vector)", "class_weight": None},
    "random_forest": {"api": "fit(sample_weight=vector)", "class_weight": None},
    "hist_gradient_boosting": {"api": "fit(sample_weight=vector)", "class_weight": None},
    "xgboost": {"api": "fit(sample_weight=vector)", "scale_pos_weight": 1.0},
    "pytorch_mlp": {"api": "BCEWithLogitsLoss(pos_weight=tensor([w_positive]), reduction='mean')"},
    "qnn": {"api": "BCEWithLogitsLoss(pos_weight=tensor([w_positive]), reduction='mean')"},
}
RANDOM_STATE_CONTRACT = {
    "dummy_prior": None,
    "fixed_l2_logistic_lbfgs": None,
    "rbf_svm_probability_false": None,
    "elastic_net_logistic_saga": "training_seed",
    "random_forest": "training_seed",
    "hist_gradient_boosting": "training_seed",
    "xgboost": "training_seed",
    "pytorch_mlp_and_qnn": "training_seed controls Python/NumPy/Torch/module initialization; minibatch generator=training_seed*10000+validation_year",
    "parallelism": "all n_jobs and numerical thread pools fixed at 1",
}
QNN_RESOURCE_POLICY = {
    "environment": {
        "python": "3.12.2",
        "pennylane": "0.45.1",
        "torch": "2.13.0",
        "device": "default.qubit",
        "shots": None,
        "dtype": "float64",
        "differentiation": "adjoint",
        "device_factory_interface": ["device_name", "wires", "shots", "backend_kwargs"],
        "future_backend_adapters": ["PennyLane-Qiskit", "PennyLane-Braket"],
        "backend_experiments_role": "secondary_robustness_only",
        "backend_constraint_adaptation_requires_documented_pre_result_amendment": True,
    },
    "synthetic_smoke_test": [
        "dependency_import_and_version_check",
        "four_and_six_qubit_device_creation",
        "batch_32_forward_and_backward_for_every_ansatz",
        "depth_1_2_3_coverage",
        "finite_outputs_and_gradients",
        "two_run_replay_max_abs_difference_at_most_1e-8",
        "checkpoint_round_trip_and_resume",
        "memory_and_runtime_estimate",
    ],
    "maximum_cumulative_wall_minutes_per_fold_fit": 120,
    "maximum_total_cpu_hours": 240,
    "maximum_total_fit_attempts": 240,
    "checkpoint_every_epochs": 5,
    "checkpoint_contents": ["weights", "optimizer_state", "epoch", "rng_states", "configuration", "fold_id", "preprocessing_and_pca_hashes", "software_versions", "device"],
    "resume_limit": 1,
    "fresh_retry_limit": 1,
    "fresh_retry_only_for_documented_infrastructure_failure_without_valid_checkpoint": True,
    "retry_may_change_methodology": False,
    "technical_infeasibility_status": "QNN_TECHNICALLY_INFEASIBLE",
    "technical_infeasibility_conditions": [
        "smoke_test_requires_model_definition_change",
        "deterministic_replay_tolerance_failure",
        "repeated_nan_inf_or_runtime_error",
        "required_fit_exceeds_120_cumulative_minutes",
        "incomplete_required_q1_or_q2_fold_set",
        "240_attempt_or_240_cpu_hour_limit_reached",
    ],
    "post_hoc_simplification_allowed": False,
}
pd.Series(QNN_RESOURCE_POLICY, name="policy")

## 6. Model selection, stability, calibration i threshold

Kandydat jest technicznie ważny tylko po utworzeniu skończonych predykcji dla wszystkich wymaganych folds. Fold dispersion i seed dispersion są report-only. Jeśli QNN jest technically infeasible, nie uczestniczy w rankingu, lecz nie unieważnia benchmarku classical/MLP.

In [ ]:
SEED_AGGREGATION = {
    "coarse_and_refinement_seed": 20260818,
    "confirmation_seeds": [20260819, 20260820],
    "stochastic_families": ["elastic_net_logistic", "random_forest", "hist_gradient_boosting", "xgboost", "pytorch_mlp", "qnn"],
    "deterministic_exceptions": {"dummy_prior": "single unweighted prior fit", "fixed_l2_logistic": "single deterministic lbfgs fit per weight mode", "rbf_svm": "single deterministic probability=False fit per configuration"},
    "oof_observation_raw_score": "arithmetic mean of aligned raw scores from seeds 20260818, 20260819, 20260820",
    "primary_metric": "compute pooled OOF PR-AUC exactly once on averaged raw scores; this value enters ranking",
    "external_and_test": "refit stochastic representative with all three seeds, arithmetic-mean raw scores, then apply its frozen Platt calibrator",
    "average_probabilities_instead_of_raw_scores_allowed": False,
}
RAW_SCORE_INTERFACE = {
    "fixed_l2_logistic": "decision_function(X)",
    "elastic_net_logistic": "decision_function(X)",
    "rbf_svm": "decision_function(X), probability=False",
    "hist_gradient_boosting": "decision_function(X)",
    "xgboost": "predict(X, output_margin=True) native margin",
    "random_forest": "logit(clip(predict_proba(X)[:,1],1e-7,1-1e-7))",
    "pytorch_mlp": "direct scalar output logit",
    "qnn": "direct scalar output logit",
    "dummy_prior": "explicit exception: logit(clip(train-prior predict_proba[:,1],1e-7,1-1e-7)); no seed ensemble",
}
SELECTION_POLICY = {
    "primary_rule": "maximum_pooled_oof_pr_auc_2015_2020",
    "stability_policy": "report_only_except_technical_invalidity",
    "required_stability_reporting": ["fold_pr_auc", "mean", "sample_sd", "minimum", "year_prevalence_comparison", "seed_dispersion"],
    "tie_definition": "equal_after_rounding_primary_metric_to_6_decimals",
    "tie_break_order": [
        "smaller_feature_block",
        "simpler_family_dummy_fixed_lr_elastic_lr_svm_histgb_xgboost_rf_pytorch_mlp_qnn",
        "no_class_weight",
        "fewer_parameters",
        "lexicographic_configuration_id",
    ],
    "calibration": {"input": "final seed-averaged pooled OOF raw scores", "method": "unweighted one-dimensional LogisticRegression Platt map", "constructor": {"penalty": None, "solver": "lbfgs", "tol": 1e-8, "fit_intercept": True, "class_weight": None, "random_state": None, "max_iter": 2000, "n_jobs": 1}, "one_calibrator_per_family_representative": True, "method_selection_allowed": False},
    "threshold": {"rule": "maximize_F1_on_the_same_calibrated_pooled_OOF_used_for_model_selection", "tie": "higher_threshold", "independent_generalization_estimate": False, "role": "operating_point_selection_only"},
    "calibration_or_threshold_may_change_model_ranking": False,
}
pd.Series(SELECTION_POLICY, name="pre_registered_rule")

## 7. Interpretability i sensitivity

Interpretowani są czterej CV-frozen representatives: najlepszy model liniowy, tree/boosting, MLP i QNN, jeśli jest technically feasible. Interpretability nie uczestniczy w model selection i nie ma interpretacji przyczynowej.

In [ ]:
INTERPRETABILITY_POLICY = {
    "common": {
        "method": "grouped_permutation_importance",
        "data": "OOF_validation_rows_2015_2020_only",
        "permutation_scope": "inside_each_year_fold_before_preprocessing",
        "raw_value_and_derived_missing_indicator_move_together": True,
        "repetitions": 20,
        "outputs": ["absolute_pr_auc_drop", "relative_pr_auc_drop", "L_D_R_block_permutation"],
        "ci": "clustered_bootstrap_by_economic_group_id",
    },
    "linear": ["standardized_coefficients", "odds_ratios", "signs", "fold_and_seed_stability", "separate_missing_indicators"],
    "tree_boosting": {"methods": ["common_permutation", "interventional_TreeSHAP"], "train_background_max": 512, "oof_rows_per_fold_max": 500, "impurity_importance_role": "diagnostic_only", "kernel_shap_fallback": False},
    "mlp": {"methods": ["common_permutation", "Integrated_Gradients_on_logit"], "steps": 64, "baseline": "zero_after_preprocessing_C", "oof_rows_per_fold_max": 200, "completeness_error_max": 1e-4},
    "qnn": {"methods": ["common_permutation_in_original_17_features", "PCA_loadings_and_explained_variance", "encoded_PCA_input_sensitivity", "fold_seed_stability", "structural_ablations"], "oof_rows_per_fold_max": 100, "full_quantum_explanation_claim_allowed": False},
    "may_change_model_or_feature_selection": False,
}
ROBUSTNESS_POLICY = {
    "pipeline": ["B_without_missing_indicators", "complete_case", "no_winsorization", "purged_economic_group_cv", "sparse_row_available_features_at_least_11_of_17"],
    "label": ["deterioration_score_at_least_2", "deterioration_score_at_least_4", "operating_performance_max_D1_D2_alternative_score_at_least_3"],
    "retune_hyperparameters": False,
    "may_change_primary_selection": False,
}
ERROR_SLICES = ["feature_year", "research_sector", "time_t_size_quartile", "x_t_status", "available_feature_count", "xbrl_availability", "economic_group_size", "false_positive", "false_negative"]
MINIMUM_REPORTED_SLICE_N = 30
pd.DataFrame([{"family": key, "policy": json.dumps(value, default=str)} for key, value in INTERPRETABILITY_POLICY.items()])

## 8. One-shot validation i drugi freeze-gate przed testem

Validation 2021–2022 może zostać otwarte dopiero po formalnym model-stage freeze. Po jego odczycie nie wolno zmieniać model family, hyperparameters, ansatz, feature block, preprocessing, calibration ani threshold i nadal nazywać pipeline niezależnie zwalidowanym. Test pozostaje zamknięty do drugiego freeze-gate.

In [ ]:
FAMILY_REPRESENTATIVE_ROSTER = {
    "families": ["dummy_prior", "fixed_l2_logistic", "elastic_net_logistic", "rbf_svm", "random_forest", "hist_gradient_boosting", "xgboost", "pytorch_mlp", "qnn_if_technically_feasible"],
    "selection_data": "frozen temporal CV OOF 2015-2020 only",
    "one_representative_per_family": True,
    "representative_frozen_fields": ["feature_block", "configuration_id_and_hyperparameters", "seed_ensemble_or_deterministic_exception", "raw_score_interface", "Platt_calibrator_parameters", "max_F1_threshold"],
    "global_winner_role": "primary external validation and test result",
    "other_representatives_role": "pre_registered secondary comparisons",
    "validation_may_remove_or_replace_test_roster_members": False,
    "technical_failure_after_roster_lock": "retain roster entry and report failed status; do not silently remove or replace",
    "qnn_infeasibility_deadline": "must be determined and recorded before external validation opens",
}
VALIDATION_AND_TEST_POLICY = {
    "external_validation": {
        "role": "one_shot_no_tune",
        "refits": [
            {"prediction_year": 2021, "train_years": [2011, 2019], "embargo_year": 2020, "exact_target_available_at_cutoff": True},
            {"prediction_year": 2022, "train_years": [2011, 2020], "embargo_year": 2021, "exact_target_available_at_cutoff": True},
        ],
        "may_tune_or_change_methodology": False,
        "evaluate_locked_family_roster": True,
        "global_winner_primary_all_others_secondary": True,
        "may_remove_models_from_test_roster": False,
    },
    "second_freeze_gate": {
        "occurs_after_one_shot_validation_and_before_test_access": True,
        "requirements": ["hash_validation_report_and_predictions", "audit_against_model_stage_preregistration", "confirm_no_methodological_changes", "record_technical_deviations_and_qnn_status"],
        "allowed_verdicts": ["MODEL PIPELINE v1 TEST-READY UNCHANGED", "MODEL PIPELINE v1 NOT TEST-READY"],
        "test_may_open_only_after_committed_test_ready_manifest": True,
    },
    "method_change_after_validation": {
        "validation_declared_spent": True,
        "new_explicit_methodology_version_required": True,
        "test_remains_closed": True,
    },
    "test_refits_if_unchanged_test_ready": [
        {"prediction_year": 2023, "training_pool_years": [2011, 2021], "embargo_year": 2022, "exact_target_available_at_cutoff": True},
        {"prediction_year": 2024, "training_pool_years": [2011, 2022], "embargo_year": 2023, "exact_target_available_at_cutoff": True},
    ],
    "test_invariants": ["entire_pre_validation_family_roster_retained", "global_winner_primary_others_secondary", "refit_preprocessing_from_zero", "fixed_model_family", "fixed_hyperparameters", "fixed_ansatz", "fixed_feature_block", "fixed_seed_ensemble", "average_raw_scores_before_calibration", "same_frozen_Platt_calibrator_per_representative", "same_frozen_threshold_per_representative", "test_2023_labels_never_train_test_2024"],
}
pd.DataFrame(VALIDATION_AND_TEST_POLICY["test_refits_if_unchanged_test_ready"] + VALIDATION_AND_TEST_POLICY["external_validation"]["refits"])

## 9. Mapowanie eksperymentu na strukturę pracy

In [ ]:
THESIS_MAPPING = [
    ("Frozen feature blocks i brak performance-driven selection", "4.3"),
    ("QNN PCA 4/6, scaling i angle mapping", "4.3"),
    ("Registry modeli; XGBoost; jawne pominięcia LDA/DT/LightGBM", "4.4"),
    ("Classical main search i conditional refinement", "4.4"),
    ("Kontrolowane QNN Stage Q1 i full-configuration Stage Q2", "4.4"),
    ("QNN resource i infeasibility policy", "4.4"),
    ("Temporal CV, ranking, seedy i stability reporting", "4.5"),
    ("Calibration, threshold i clustered inference", "4.5"),
    ("Interpretability protocol", "4.5"),
    ("Charakterystyka próby i klas z notebooków 01–04", "5.1"),
    ("Classical, MLP i QNN OOF/external results", "5.2"),
    ("Controlled ansatz comparison", "5.2, 5.3"),
    ("Feature, preprocessing, sample i label robustness", "5.3"),
    ("QNN structural sensitivity", "5.3"),
    ("Error analysis i fold/seed instability", "5.4"),
    ("Coefficients, TreeSHAP, MLP IG i common permutation", "5.4"),
    ("Defensywna interpretacja QNN i technical infeasibility", "5.4"),
    ("Validation/test freeze gates i ograniczenia generalizacji", "5.4"),
]
pd.DataFrame(THESIS_MAPPING, columns=["element_eksperymentu", "sekcja_pracy"])

## 10. Preregistration gate

Ten notebook nie zamraża automatycznie etapu modelowego. Po review jego specyfikacja musi zostać przeniesiona do wersjonowanej konfiguracji, manifestu i policy tests. Dopiero osobny formalny freeze może autoryzować trening na train 2011–2020.

In [ ]:
assert EXPERIMENT_CONTRACT["model_training_performed"] is False
assert EXPERIMENT_CONTRACT["feature_blocks"] == ["L", "L+D", "L+D+R"]
assert QNN_STAGE_Q1["may_select_final_ansatz"] is True
assert QNN_STAGE_Q1["stage_q2_may_change_ansatz"] is False
assert len(QNN_ANSATZES) == 3 and len(QNN_STAGE_Q2) == 4
assert REFINEMENT_POLICY["validation_or_test_may_activate"] is False
assert INTERPRETABILITY_POLICY["may_change_model_or_feature_selection"] is False
assert ROBUSTNESS_POLICY["retune_hyperparameters"] is False
assert VALIDATION_AND_TEST_POLICY["second_freeze_gate"]["test_may_open_only_after_committed_test_ready_manifest"] is True
assert QNN_RESOURCE_POLICY["post_hoc_simplification_allowed"] is False
assert {row["id"] for row in MODEL_REGISTRY} == {"dummy_prior", "fixed_l2_logistic", "elastic_net_logistic", "rbf_svm", "random_forest", "hist_gradient_boosting", "xgboost", "pytorch_mlp"}
elastic_space = next(row["space"] for row in MODEL_REGISTRY if row["id"] == "elastic_net_logistic")
assert elastic_space["penalty"] == ["elasticnet"] and elastic_space["solver"] == ["saga"]
assert ELASTIC_NET_CONSTRUCTOR["penalty"] == "elasticnet" and ELASTIC_NET_CONSTRUCTOR["solver"] == "saga"
mlp_space = next(row["space"] for row in MODEL_REGISTRY if row["id"] == "pytorch_mlp")
assert "weight_decay" in mlp_space and "epochs" in mlp_space and "alpha" not in mlp_space and "max_iter" not in mlp_space
assert PYTORCH_MLP_CONTRACT["loss"]["reduction"] == "mean"
assert SEED_AGGREGATION["coarse_and_refinement_seed"] == 20260818 and SEED_AGGREGATION["confirmation_seeds"] == [20260819, 20260820]
assert RAW_SCORE_INTERFACE["xgboost"].endswith("native margin")
assert FAMILY_REPRESENTATIVE_ROSTER["validation_may_remove_or_replace_test_roster_members"] is False
assert PCA_QNN_CONTRACT["includes_missing_indicators"] is True and PCA_QNN_CONTRACT["pca"]["svd_solver"] == "full" and PCA_QNN_CONTRACT["pca"]["whiten"] is False
assert {name: package["trainable_parameters_q4_depth2_including_head"] for name, package in QNN_ARCHITECTURE_PACKAGES.items()} == {"ROT_CNOT_RING": 29, "RY_RZ_CZ_BRICKWORK": 21, "RY_CRX_RING": 21}
assert DIAGNOSTIC_BUDGET["total"] == 12 and DIAGNOSTIC_BUDGET["included_in_primary_ranking_budget"] is False
assert all(canonical_sha256(materialized_lists[name]) == expected for name, expected in candidate_registry["list_hashes"].items())
TECHNICAL_FREEZE_GATE_VERDICT = "MODEL STAGE READY TO FREEZE"
print(TECHNICAL_FREEZE_GATE_VERDICT)
print("MODEL TRAINING PERFORMED: FALSE")
print("EXTERNAL VALIDATION 2021–2022 OPENED ANALYTICALLY: FALSE")
print("TEST 2023–2024 USED: FALSE")
print("UNRESOLVED DECISIONS REQUIRING APPROVAL: NONE")